### Rag PipeLine -> Data ingestion to vector DB PipeLine

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

# load HF_TOKEN (and any other vars) from ../.env
load_dotenv()

/tmp/ipykernel_16169/10798220.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/home/emandeveloper/emanData/webDev/RagComplete/Rag1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Read all the pdf inside directory

def process_all_pdfs(pdf_directory):

    all_documents=[]
    pdf_dir=Path(pdf_directory)

    # find all pdf file recursively inside the directory
    # ** mean search all folder and sub folder recursively
    # pdf_files is now have list of all pdf in pdf_directory and sub directory of pdf_directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            # Load the PDF file using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:

            #    for adding extra information in meta data like here we add file name and type of file

               doc.metadata['source_file'] = pdf_file.name 
               doc.metadata['file_type']='pdf'

            # extend add the add pdf document in one list
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files in ../data
Processing ../data/file-sample_150kB.pdf...
Loaded 4 pages
Processing ../data/iso27001.pdf...
Loaded 26 pages
Total documents loaded: 30


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit

In [4]:
# now we convert our documents into chunks

# chunk_size is the size of each chunk in number of characters
# chunks_overlap is the number of characters that overlap between chunks
# overlap chunks example
# Chunk 1:    1 -------- 1000
# Chunk 2:         801 -------- 1800
# Chunk 3:               1601 ------- 2500
def split_documents(documents, chunk_size=1000, chunks_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunks_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_documents=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_documents)} chunks")

    # show example of chunks

    if split_documents:
        print("// Example of a chunk:")
        print(f'content: {split_documents[0].page_content[:200]}...')  # Print first 200 characters of the first chunk
        print(f'chunk metadata: {split_documents[0].metadata}')  # Print metadata of the first chunk

    return split_documents


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

split 30 documents into 85 chunks
// Example of a chunk:
content: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nulla...
chunk metadata: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit

### Embeding and vector Store Db

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbedingManager:

    # model is used to convert text into vectors
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Load successfully. Embeding dimention: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def genrate_embedding(self, texts: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model is not loaded. Please call _load_model() first.")
            
        print(f"Generating embedding for {len(texts)} characters")  # Print first 50 characters of the text
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding shape: {embedding.shape}")  # Print the shape of the generated embedding
        return embedding

    # def get_embedding_dimension(self) -> int:
    #     if not self.model:
    #         raise ValueError("Model is not loaded. Please call _load_model() first.")
    #     return self.model.get_sentence_embedding_dimension()


# initialize the EmbedingManager with the desired model
embedding_manager = EmbedingManager(model_name="all-MiniLM-L6-v2")
# it run the _load_model() method and load the model into memory
embedding_manager
       

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4056.46it/s]


Model Load successfully. Embeding dimention: 384


/tmp/ipykernel_16169/3081748277.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Load successfully. Embeding dimention: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [8]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client=None
        self.collection=None
        self._initalize_store()

    def _initalize_store(self):
        # Initialize ChromaDB client and collection
        try:
            # create persist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"}
            )

            print(f"Collection name: {self.collection.name}")
            print(f"Collection metadata: {self.collection.metadata}")
           
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for insertion
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate a unique ID for each document
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)  # collect the id so the ids list is not empty

            # prepared metadata
            metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
            metadata['doc_index'] = i  # Add index to metadata for reference
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)

            # document content 
            documents_texts.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB

        # Add documents to the collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Collection now contains {self.collection.count()} documents.")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Collection name: pdf_documents
Collection metadata: {'description': 'Collection of PDF document embeddings'}


In [9]:
# convert text to embeding
texts=[doc.page_content for doc in chunks]
texts

['Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet\ntortor quis risus auctor condimentum. Morbi in ullamcorper elit. Nulla iaculis tellus sit amet\nmauris tempus fringilla.\nMaecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.\n\uf0b7 Maecenas non lorem quis tellus placerat varius. \n\uf0b7 Nulla facilisi

In [10]:
# genrate embedding for the text chunks
embeddings = embedding_manager.genrate_embedding(texts)

Generating embedding for 85 characters


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.19s/it]

Generated embedding shape: (85, 384)


In [11]:
# store in vector data base
vector_store.add_documents(chunks, embeddings)

Adding 85 documents to the vector store...
Successfully added 85 documents to the vector store.
Collection now contains 595 documents.


### Retrival pip line from vector

In [13]:
class RagRetrival:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbedingManager):

        '''
        Initialize the RAG retrieval system with a vector store and an embedding manager.
        vector_store: vector store contain document embedings and metadata
        emedding_manager: manager for genrating query embedings'''

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int=5,score_threshold:float=0.0)-> List[Dict[str,Any]]:

        '''
        Retrieve relaivant document for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold:Minimum similarity score threshold

        Returns:
                List of dictionary containing retrieved documents and metadata
        '''
        print(f"Retriving document for query: {query}")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # Genrate query embeding
        query_embedding=self.embedding_manager.genrate_embedding([query])[0]

        # search in vector store

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process resutl
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i, (doc_id, document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score=1-distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents after filtring")

            else:
                print("no document found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrival: {e}")
            return []


rag_retrivver=RagRetrival(vector_store,embedding_manager)


In [14]:
rag_retrivver

In [15]:
rag_retrivver.retrieve("about information security")

Retriving document for query: about information security
Top k: 5, Score threshold: 0.0
Generating embedding for 1 characters


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.79it/s]

Generated embedding shape: (1, 384)
Retrieved 5 documents after filtring


[{'id': '435a032d-4caa-4cba-8c22-42baf4a10847',
  'content': 'assignment of current employees; or the hiring or contracting of competent persons.\n7.3  Awareness\nPersons doing work under the organization’s control shall be aware of:\na) the information security policy;\nb) their contribution to the effectiveness of the information security management system, including \nthe benefits of improved information security performance; and\nc) the implications of not conforming with the information security management system \nrequirements.\n7.4  Communication\nThe organization shall determine the need for internal and external communications relevant to the \ninformation security management system including:\na) on what to communicate;\nb) when to communicate;\nc) with whom to communicate;\nd) how to communicate.\n7.5  Documented information\n7.5.1  General\nThe organization’s information security management system shall include:\na) documented information required by this document; and\n   

### Query retrivel pip line

In [16]:
#Simple Rag pipeline with groq llm

from langchain_groq import ChatGroq

# initialize the GroqChat model with your API key
groq_api_key = os.getenv("groq_api")

if not groq_api_key:
    raise ValueError("Groq API key not found in environment variables. Please set 'groq_api' in your .env file.")

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",  # current Groq production model (gemma2-9b-it was decommissioned)
    temperature=0.1,
    max_tokens=1000
)

def rag_simple(query,retriever,llm,top_k=3):

    # retrieve the relavent context
    results=retriever.retrieve(query,top_k=top_k)

    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant documents found."

    # genrate the answer with llm

    prompt=f"""Use the following context to answer the question consisely.
    Context:
    {context}
    Question: {query}
    Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])

    return response.content



In [26]:
answer=rag_simple("information security?",rag_retrivver,llm)

answer

Retriving document for query: information security?
Top k: 3, Score threshold: 0.0
Generating embedding for 1 characters


Batches: 100%|██████████| 1/1 [00:00<00:00, 118.95it/s]

Generated embedding shape: (1, 384)
Retrieved 3 documents after filtring


"The organization's information security management system includes: \n- assignment of competent persons, \n- awareness of information security policy and implications, \n- internal and external communications, and \n- documented information."

### Enhance Rag pip line

In [17]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("information security", rag_retrivver, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retriving document for query: information security
Top k: 3, Score threshold: 0.1
Generating embedding for 1 characters


Batches: 100%|██████████| 1/1 [00:00<00:00, 103.61it/s]

Generated embedding shape: (1, 384)
Retrieved 3 documents after filtring


Answer: is considered in the design of processes, information systems, and controls, and risks are adequately managed.
Sources: [{'source': 'iso27001.pdf', 'page': 4, 'score': 0.3389096260070801, 'preview': 'risks are adequately managed.\nIt is important that the information security management system is part of and integrated with the \norganization’s processes and overall management structure and that information security is considered \nin the design of processes, information systems, and controls. It i...'}, {'source': 'iso27001.pdf', 'page': 4, 'score': 0.3389096260070801, 'preview': 'risks are adequately managed.\nIt is important that the information security management system is part of and integrated with the \norganization’s processes and overall management structure and that information security is considered \nin the design of processes, information systems, and controls. It i...'}, {'source': 'iso27001.pdf', 'page': 4, 'score': 0.3389096260070801, 'preview': 'risks are adeq

In [20]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retrivver, llm)
result = adv_rag.query("information security", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retriving document for query: information security
Top k: 3, Score threshold: 0.1
Generating embedding for 1 characters


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.30it/s]

Generated embedding shape: (1, 384)
Retrieved 3 documents after filtring
Streaming answer:
Use the following context to answer the question concisely.
Context:
risks are adequately managed.
It is important that the information security management system is part of and integrated with the 
organization’s processes and overall management structure and that information security is considered 
in the design of p

rocesses, information systems, and controls. It is expected that an information security 
management system implementation will be scaled in accordance with the needs of the organization.
This document can be used by internal and external parties to assess the organization's ability to meet 
the organization’s own information security requirements.
The order in which requirements are presented in this document does not reflect their importance 
or imply the order in which they are to be implemented. The list items are enumerated for reference 
purpose only.
ISO/IEC 27000 describes the overview and the vocabulary of information security management

risks are adequately managed.
It is important that the information security management system is part of and integrated with the 
organization’s processes and overall management structure and that information security is considered 
in the design of processes, information systems, and controls. It is expected that an information security 
man